In [18]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import os
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [13]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')


Shape of X_train: (454902, 30)
Shape of X_test: (56962, 30)


In [23]:
print("Training Logistic Regression..")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

print("\nLogistic Regression Results:")
print(classification_report(y_test, lr_preds, target_names=['Legit', 'Fraud']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")


Training Logistic Regression..


/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sk


Logistic Regression Results:
              precision    recall  f1-score   support

       Legit       1.00      0.97      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

ROC-AUC: 0.9699


/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extm

In [19]:
# Building Parameter grid for Random Search CV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1, 5, 10]
}

In [20]:
xgb = XGBClassifier(random_state=42, eval_metric='logloss')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=20,          # tries 20 random combinations
    scoring='roc_auc',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1           # uses all CPU cores
)

search.fit(X_train, y_train)

print(f"\nBest parameters: {search.best_params_}")
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1, subsample=1.0; total time=   4.1s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1, subsample=1.0; total time=   4.2s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1, subsample=1.0; total time=   4.2s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1, subsample=1.0; total time=   4.3s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1, subsample=1.0; total time=   4.3s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=200, scale_pos_weight=10, subsample=0.8; total time=   5.9s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=200, scale_pos_weight=10, subsample=0.8; tot

/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=  10.1s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=   9.8s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   6.3s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   6.7s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=300, scale_pos_weight=5, subsample=0.8; total time=  10.4s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   6.8s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   6.6s
[CV] END colsample_bytree=1.0, learning_rate=0.1, m

In [22]:
best_model = search.best_estimator_

preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)[:, 1]

print("Tuned XGBoost Results:")
print(classification_report(y_test, preds, target_names=['Legit', 'Fraud']))
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")

Tuned XGBoost Results:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.67      0.88      0.76        98

    accuracy                           1.00     56962
   macro avg       0.84      0.94      0.88     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9821


Training Logistic Regression..


/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sk


Logistic Regression Results:
              precision    recall  f1-score   support

       Legit       1.00      0.97      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

ROC-AUC: 0.9699


/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/refgolecruz/Desktop/COMP648/fraud-detection/.venv/lib/python3.9/site-packages/sklearn/utils/extm

In [24]:
joblib.dump(best_model, '../models/fraud_model_tuned.joblib')
print("Tuned model saved!")

Tuned model saved!
